## Exploring the Vector stores -chroma DB and Qrant with LlamaIndex

#### Setup and import common libraries

In [ ]:
from llama_index.core import Document, Settings, VectorStoreIndex

from llama_index.core.node_parser import SentenceSplitter, TokenTextSplitter


from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding


from pathlib import Path
import os
import time
import warnings
from dotenv import load_dotenv

warnings.filterwarnings("ignore")


In [ ]:
#Loaading the env and Setting up the LlamaIndex Settings

load_dotenv()

Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0.2)

Settings.embed_model=OpenAIEmbedding(model="text-embedding-3-small", dimensions=1536)

Settings.chunk_size=728
Settings.chunk_overlap=100

Settings.text_splitter=TokenTextSplitter(chunk_size=Settings.chunk_size,
                                        chunk_overlap=Settings.chunk_overlap,
                                        separator=".",
                                        )


In [ ]:
from llama_parse import LlamaParse

data=Path("./data")

research_file_dir = data/"research_papers"

for file in list(research_file_dir.glob("*.pdf")):
    print(f"file name : {file.name} and path : {file.parent}")

In [ ]:
file = research_file_dir/"attention_paper.pdf"
if file.exists():
    True

In [ ]:
#Initializing the llamaparser

file_parser=LlamaParse(show_progress = True, result_type="markdown")
if file.exists():
    documents = file_parser.load_data(file_path=file,extra_info={'filename':str(file)})
    print(f"len of documents : {len(documents)}")
else:
    print(f"directory {file} not found")

In [ ]:
### We don't have to call the text spitter to get the chunks as it will taken care at the time of indexing
nodes = Settings.text_splitter.get_nodes_from_documents(documents=documents)
print(f"no of nodes created :{len(nodes)}")

#### First Let's check the in-moemory indexing using LlamaIndex

In [ ]:
st_time = time.time()
vector_store = VectorStoreIndex.from_documents(documents=documents,show_progress=True)
total_time=time.time() - st_time

print(f"No of index created {(vector_store)} and time elapsed {total_time}")

** Note **: If the transformation is not passed in the function call then the splitteris not taken.

In [ ]:
print(vector_store.__dict__.keys())

for k,v in  vector_store.__dict__.items():
    print(f"{k} : {v}")

In [ ]:
print(len(vector_store._index_struct.nodes_dict))

In [ ]:
query  = "What is attention according to transformers?"

vector_engine=vector_store.as_query_engine()

ans1=vector_engine.query(query)

In [ ]:
print(len(ans1.source_nodes[1].text))

### Exploring ChromaDB 

In [ ]:
import chromadb

from llama_index.vector_stores.chroma import ChromaVectorStore

from llama_index.core import StorageContext


In [ ]:
chroma_client = chromadb.EphemeralClient()

collection_name = "llama_index_pract"

chroma_collection = chroma_client.create_collection(name=collection_name)

chroma_store = ChromaVectorStore(chroma_collection=chroma_collection)

storage_context = StorageContext.from_defaults(vector_store=chroma_store)

st_time = time.time()
chroma_index = VectorStoreIndex.from_documents(documents=documents,
                                               storage_context=storage_context,
                                               show_progress=True)

time_taken=time.time()-st_time

print(f"Chroma vector is used to create the index within the time {time_taken}")

Chroma Features:

Ease of use: Zero configuration for local dev

Metadata filtering: WHERE clause support

Distance metrics: Cosine (default), L2, IP

Persistence: Optional disk storage

Scales to: ~1M vectors comfortably

In [ ]:
#To check how the persistant works

chroma_client=chromadb.PersistentClient(path="./chroma")

chroma_collection = chroma_client.get_or_create_collection(name="llama_index_practice")

chroma_store = ChromaVectorStore(chroma_collection=chroma_collection)

storage_context = StorageContext.from_defaults(vector_store=chroma_store)

chroma_index = VectorStoreIndex.from_documents(
    documents=nodes,
    storage_context=storage_context,
    show_progress=True,
    transformations=[Settings.text_splitter]
)

In [ ]:
print(chroma_index.__dict__)

In [ ]:
print(len(chroma_index._index_struct.nodes_dict))

In [ ]:
query  = "What is attention according to transformers?"

chroma_engine=chroma_index.as_query_engine()

ans1=chroma_engine.query(query)

In [ ]:
print(len(ans1.source_nodes[1].text))

###  Explaoring Qdrant vector store

In [ ]:
from llama_index.vector_stores.qdrant import QdrantVectorStore

from qdrant_client import QdrantClient


from llama_index.core import StorageContext


In [ ]:
qclient=QdrantClient(location=':memory:')

collection_name = "llamaIndex_practice"
qdrant_store=QdrantVectorStore(client=qclient,
                               collection_name=collection_name)

storage_context  =StorageContext.from_defaults(vector_store=qdrant_store)

qdrant_index = VectorStoreIndex.from_documents(documents = documents,
                                               storage_context=storage_context,
                                               show_progress = True )




In [ ]:
for k,v in  qdrant_index.__dict__.items():
    print(f"{k} : {v}")

In [ ]:
print(qdrant_index._index_struct.nodes_dict)

In [ ]:
query= "what is attention mechanism?"

qdrant_engine=qdrant_index.as_query_engine()

ans1 = qdrant_engine.query(query)

In [ ]:
print(ans1)

### ML Engineering Note: Vector Database Comparison

| Feature | SimpleVectorStore | Chroma | Qdrant |
|---------|------------------|--------|--------|
| **Setup** | Built-in | Easy | Moderate |
| **Scale** | <10k vectors | ~1M vectors | 10M+ vectors |
| **Speed** | Moderate | Fast | Very Fast |
| **Filtering** | Basic | Good | Excellent |
| **Hybrid Search** | No | No | Yes |
| **Cloud Option** | No | Planned | Yes |
| **Best For** | Prototyping | Small-medium apps | Production |

**Recommendation**: 
- Prototyping: SimpleVectorStore or Chroma
- Production (<1M docs): Chroma or Qdrant
- Production (>1M docs): Qdrant, Pinecone, or Weaviate

### 🎯 ML Engineering Note: Embedding Selection

**OpenAI (text-embedding-3-small):**
- ✅ State-of-the-art quality
- ✅ Variable dimensions (256-1536)
- ✅ No model hosting needed
- ❌ Costs per API call
- ❌ Data leaves your infrastructure

**HuggingFace (all-MiniLM-L6-v2):**
- ✅ Free and open-source
- ✅ Runs locally (data privacy)
- ✅ Fast inference (especially with GPU)
- ❌ Lower quality than OpenAI
- ❌ Fixed dimensions (384)
- ❌ Requires model hosting

**Recommendation**: 
- Development: OpenAI (fast iteration)
- Production (high quality needed): OpenAI
- Production (cost-sensitive, privacy): HuggingFace + GPU

### Loading from storage.

In [ ]:
from llama_index.core import load_index_from_storage
persist_dir = "./storage"

vector_store.storage_context.persist(persist_dir=persist_dir)

storage_context_loaded=storage_context.from_defaults(persist_dir=persist_dir)

loded_index=load_index_from_storage(storage_context_loaded)

In [ ]:
query="what are dataset used in experiment?"

index_engine = loded_index.as_query_engine(similarity_top_k=2)
ans = index_engine.query(query)

In [ ]:
print(ans)

### Streaming proccess

In [ ]:
stream_engine = qdrant_index.as_query_engine(streaming=True)

ans=stream_engine.query(query)

for text in ans.response_gen:
    print(text, end="", flush=True)

### VectorIndex Retriever

In [ ]:
from llama_index.core.retrievers import VectorIndexRetriever


retriever = VectorIndexRetriever(index = qdrant_index,
                                 streaming=True)

ans = retriever.retrieve(query)

for i, node in enumerate(ans,1):
    print(f"Node {i}")
    print(node.text)


#### Custome Query Engine Retriever

In [ ]:

from llama_index.core.query_engine import RetrieverQueryEngine
custom_retriever = RetrieverQueryEngine.from_args(retriever=retriever,response_mode='tree_summarize')

ans = custom_retriever.query(query)

In [ ]:
print(ans)

### VectorIndexAutoRetriever


##### Natural Language Metadata filtering

In [ ]:
from llama_index.core.vector_stores import VectorStoreInfo, MetadataInfo


from llama_index.core.retrievers import VectorIndexAutoRetriever


vector_store_info = VectorStoreInfo(
    content_info="Technical documentation about vector databases and embeddings",
    metadata_info=[
        MetadataInfo(
            name="topic",
            type="str",
            description="The main topic of the document (e.g., 'qdrant', 'chroma', 'embeddings')",
        ),
        MetadataInfo(
            name="difficulty",
            type="str",
            description="Difficulty level: 'beginner', 'intermediate', or 'advanced'",
        ),
        MetadataInfo(
            name="year",
            type="int",
            description="Year of publication (2023 or 2024)",
        ),
    ],
)

# Create auto-retriever
auto_retriever = VectorIndexAutoRetriever(
    qdrant_index,
    vector_store_info=vector_store_info,
    similarity_top_k=3,
)

print("✅ VectorIndexAutoRetriever configured")

In [ ]:
query_with_filter = "Tell me about beginner-level topics"

print(f"Query: {query_with_filter}\n")
print("Auto-retriever will automatically extract metadata filters from the query!\n")

retrieved = auto_retriever.retrieve(query_with_filter)

print(f"Retrieved {len(retrieved)} nodes:\n")
for i, node in enumerate(retrieved, 1):
    print(f"Node {i}:")
    print(f"  Topic: {node.metadata.get('topic')}")
    print(f"  Difficulty: {node.metadata.get('difficulty')}")
    print(f"  Score: {node.score:.4f}")
    print()

The Above cell did not retrive any nodes as we don't have any nodes matching the metadata for beginners. This is here to show the process of metadata filtering using llama_index